In [1]:
# NLP303 Assessment 3
#
# Detecting AI-generated Text
# Using Transformer-Based Classification
#
#
# Jonathan Lim - A00142089
#
# Thomas Galindo Salazar - A00129258
#
# Tibor Titusz Tarcsai - A00121308
#
#
#
# The purpose of this implementation is to demonstrate the building of a working prototype for a Classification task based on the Assessment 2 proposal.
#
#
#***********************
# HOW TO RUN THE CODE:
#***********************
#
# 1. - The notebook can run either in Jupyter Notebook or Google Colab
#    - If using Google Colab, a Google Drive account is required for data access and storage.
#      Link for running on Colab:
#
# 2. All helper functions are imported and loaded from the src folder
#

# Environment Setup and Dependencies

In [1]:
# Clones Project from Github
!git clone -b colab https://github.com/Titusz87/roberta-ai-text-detector.git

# Sets working directory for Colab
%cd /content/roberta-ai-text-detector/notebook/

Cloning into 'roberta-ai-text-detector'...
remote: Enumerating objects: 305, done.
remote: Counting objects: 100% (305/305), done.
remote: Compressing objects: 100% (171/171), done.
remote: Total 305 (delta 168), reused 211 (delta 85), pack-reused 0 (from 0)
Receiving objects: 100% (305/305), 137.49 KiB | 1.48 MiB/s, done.
Resolving deltas: 100% (168/168), done.
[Errno 2] No such file or directory: '/content/roberta-ai-text-detector/notebook/'
/home/titus/projects/roberta-ai-text-detector/notebook


In [1]:
!pip install -r ../requirements.txt
print("\n\n### Dependencies installed successfully ###")



### Dependencies installed successfully ###


# Section 1 - Load Dataset

In [3]:
# Initialises the downloader class
import sys
sys.path.append("../")

from src.utils.downloader import Downloader

downloader = Downloader()

# Downloads raw datasets from google drive
print("### Downloading Dataset from Google Drive... ###")
downloader.start_downloading_dataset()
print(f"\n\nDatasets are downloaded and and located at folder 'data/' .")

### Downloading Dataset from Google Drive... ###


Downloading...
From: https://drive.google.com/uc?id=19etkvYMcxOwKDpX_d3069YmAv1UXuZku
To: /home/titus/projects/roberta-ai-text-detector/data/humanised_first_2000.csv
100%|██████████| 6.95M/6.95M [00:01<00:00, 6.15MB/s]
Downloading...
From: https://drive.google.com/uc?id=12IAqrDtXtS6W08-Qy0fNhmmS62bGdUmS
To: /home/titus/projects/roberta-ai-text-detector/data/ai_polished_first_2000.csv
100%|██████████| 6.72M/6.72M [00:01<00:00, 5.12MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cMnmb5MBfXNDxYnx0R7xsOovywZvh7yQ
To: /home/titus/projects/roberta-ai-text-detector/data/pure_ai_first_2000.csv
100%|██████████| 7.18M/7.18M [00:01<00:00, 5.39MB/s]
Downloading...
From: https://drive.google.com/uc?id=1nJHkqxdrFbz9XNdk8tWKlxsnQ8XoLFHf
To: /home/titus/projects/roberta-ai-text-detector/data/pure_human_first_2000.csv
100%|██████████| 3.62M/3.62M [00:00<00:00, 4.74MB/s]



Datasets are downloaded and and located at folder 'data/' .


In [4]:
# Builds the dataset
from src.datasetbuilder import DatasetBuilder

builder = DatasetBuilder()

raw_dataset = builder.build_dataset(
    builder.raw_dataset_paths[0]["pure_human"],     # Label 0
    builder.raw_dataset_paths[1]["pure_ai"],        # Label 1
    builder.raw_dataset_paths[2]["ai_polished"],    # Label 2
    builder.raw_dataset_paths[3]["humanised"],      # Label 3
)
# Prints the total number of samples from the dataset
print(len(raw_dataset))

8000


# Section 2 - Text Preprocessing

In [5]:
# Drops rows where 'text' is missing
raw_dataset = raw_dataset.dropna(subset=["text"]).reset_index(drop=True)

# Prints the number of samples left after the first preprocessing step
print(len(raw_dataset))

8000


In [6]:
# Initialises the TextPreprocessor class for text cleaning
from src.utils.preprocessing import TextPreprocessor

text_preprocessor = TextPreprocessor()

raw_dataset["cleaned_text"] = raw_dataset["text"].apply(text_preprocessor.clean_text)


# Prints a sample of text
print(raw_dataset[["cleaned_text", "text"]].iloc[5850])

cleaned_text    I used Texmati 2 andouille sausage links 1/2 c...
text            I used Texmati\n2 andouille sausage links\n1/2...
Name: 5850, dtype: str


# Section 3 - Data Split

In [7]:
# Shuffles the dataset and stores features and labels
dataset_shuffled = raw_dataset.sample(frac=1, random_state=42).reset_index(drop=True)

features = dataset_shuffled['cleaned_text']
labels = dataset_shuffled['label']


# Splits the data into an initial 80-20 split
from sklearn.model_selection import train_test_split

x_train_raw, x_temp_raw, y_train, y_temp = train_test_split(
    features,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

# Split the remaining data into validation and test (10-10)
x_val_raw, x_test_raw, y_val, y_test = train_test_split(
    x_temp_raw,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Prints class distributions

print("\n * Training Labels (y_train):")

print(y_train.value_counts().sort_index())

print("\n * Validation Labels (y_val):")

print(y_val.value_counts().sort_index())

print("\n * Testing Labels (y_test):")

print(y_test.value_counts().sort_index())


 * Training Labels (y_train):
label
0    1600
1    1600
2    1600
3    1600
Name: count, dtype: int64

 * Validation Labels (y_val):
label
0    200
1    200
2    200
3    200
Name: count, dtype: int64

 * Testing Labels (y_test):
label
0    200
1    200
2    200
3    200
Name: count, dtype: int64


# Section 3 - Tokenisation (BPE)

In [8]:

"""""
In this section during tokenisation the encodings are created to suit the basemodel with 512 token limit.
"""""

# Imports tokeniser
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("fakespot-ai/roberta-base-ai-text-detection-v1")

# Creates input encodings
train_encodings = tokenizer(x_train_raw.to_list(), truncation=True, padding="max_length", max_length=512)
val_encodings = tokenizer(x_val_raw.to_list(), truncation=True, padding="max_length", max_length=512)
test_encodings = tokenizer(x_test_raw.to_list(), truncation=True, padding="max_length", max_length=512)

# Transforms encodings into Pytorch tensors
from src.dataset_wrapper import DatasetWrapper

X_train = DatasetWrapper(train_encodings, y_train)
X_val = DatasetWrapper(val_encodings, y_val)
X_test = DatasetWrapper(test_encodings, y_test)


# Prints the first sample's token IDs, mask, and label
X_test[0]

{'input_ids': tensor([    0,   487,   990, 23943, 25567,  1499,    16,    10,  1385,   341,
             7,  6190,    10, 29927,  2412, 41021,  7208,    14,  6330,     7,
         27389,     5, 29500,     9, 12786, 23943,  1809,    19,     5, 12049,
             8,  2956,  2633,    11,     5,  1406,  2898,   162,  7794, 28805,
           651, 36349, 24716, 25567,  1499,     4, 12786, 23943,  1809,    21,
            10,  4845,  9779, 26944,   280,  4373,    11,     5,  1998,  3220,
          4516,     6,    61, 12843,     5, 10875,   227,     5,  1050,     8,
         26280,   295, 15488,     9,  5772,  4845,     4,   152,  1217,    21,
          2998,   259, 39267,    30,     5,  1080,     9, 20623,  7618,   261,
            11, 40212,  4516,     6,    25,    24,    21,   450,    25, 22757,
            19,     5, 18466,  2412,  6563,    11,     5,  8618,     9,  4845,
            18, 15671,  2603, 17048,    36,  3361,  4086,    43,     8,    39,
          6594,   295, 15488,    36, 19

# Section 4- Fine-tuning - Part 1

In [9]:
"""""
After the input is set, the pretrained model from HuggingFace is prepared in the following steps in part 1:
- model loaded for further fine-tuning with optional gpu acceleration,
- a new classification head is set for a new downstream task to classify 4 classes,
- and lastly pretrained model summary is printed.
"""""
import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification

device = 'cuda' if torch.cuda.is_available() else 'cpu'


model = AutoModelForSequenceClassification.from_pretrained(
    "fakespot-ai/roberta-base-ai-text-detection-v1",
    num_labels=4,
    ignore_mismatched_sizes=True)

# Prints model summary
from torchinfo import summary

summary(model, input_size=(1, 512), dtypes=[torch.long])

I0000 00:00:1786768600.400814   28005 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at fakespot-ai/roberta-base-ai-text-detection-v1 and are newly initialized because the shapes did not match:
- classifier.out_proj.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.out_proj.weight: found shape torch.Size([2, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Layer (type:depth-idx)                                            Output Shape              Param #
RobertaForSequenceClassification                                  [1, 4]                    --
├─RobertaModel: 1-1                                               [1, 512, 768]             --
│    └─RobertaEmbeddings: 2-1                                     [1, 512, 768]             --
│    │    └─Embedding: 3-1                                        [1, 512, 768]             38,603,520
│    │    └─Embedding: 3-2                                        [1, 512, 768]             768
│    │    └─Embedding: 3-3                                        [1, 512, 768]             394,752
│    │    └─LayerNorm: 3-4                                        [1, 512, 768]             1,536
│    │    └─Dropout: 3-5                                          [1, 512, 768]             --
│    └─RobertaEncoder: 2-2                                        [1, 512, 768]             --
│    │    └─ModuleList: 3-6 

In [10]:
"""""
Change the flag to "True" only to proceed fine-tuning, otherwise in the next coming sectoin it loads the save model config from previous session.
"""""

IS_FINE_TUNING_ON = False

In [11]:
# Fine-tuning - Part 2

# REFERENCE: https://huggingface.co/transformers/v3.2.0/custom_datasets.html

"""""
In part 2 the model is prepared for the actual fine-tuning process as follows:
- pretrained model is loaded into device (whether cpu or gpu if available),
- train_loader is initialized to stream and shuffle the dataset in mini-batches of 16,
- the AdamW optimizer is instantiated with a 5e-5 learning rate to update model parameters.

- the fine tuning process goes through 3 epochs where:
                        - total loss is zerod to..
                        -
                        -
                        -
"""""
if (IS_FINE_TUNING_ON):

    from src.val_evaluator import validate_model

    model.to(device)
    model.train()

    train_loader = DataLoader(X_train, batch_size=16, shuffle=True)
    val_loader = DataLoader(X_val, batch_size=16, shuffle=False)
    optimiser = AdamW(model.parameters(), lr=2e-5)     # lowered from 5e-5 REFERENCE:https://discuss.huggingface.co/t/opinion-training-argument-fine-tuning-mlm-roberta/135013
    
    print("Fine tuning started...")

    batch_losses = []
    train_losses = []
    val_losses = []


    # Early stopping variables
    best_val_loss = float("inf")
    patience = 0
    patience_threshold = 3
    number_of_epochs=10

    for epoch in range(number_of_epochs):
        
        total_loss = 0.0

        for batch_idx, batch in enumerate(train_loader):
            optimiser.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

            loss = outputs.loss
            batch_losses.append(loss.item())

            loss.backward()
            optimiser.step()

            total_loss += loss.item()

        # Average training loss
        avg_train_loss = total_loss / len(train_loader)

        # Validation loss
        avg_val_loss, best_val_loss, patience = validate_model(
            patience, 
            best_val_loss, 
            model,
            tokenizer, 
            val_loader, 
            device)

        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)

        print(
            f"Epoch {epoch+1}/{number_of_epochs} | "
            f"Training Loss: {avg_train_loss:.4f} | "
            f"Validation Loss: {avg_val_loss:.4f}"
        )

        if patience >= patience_threshold:
            print("Early stopping triggered.")
            break

    model.eval()
    print("Fine tuning is completed.")

    from src.utils.loss_logger import save_log

    save_log( train_losses,
              val_losses, 
              batch_losses,
              number_of_epochs,
              optimiser )

In [ ]:
import os

# With fine-tuning true flag, it loads the highest-performing archived model during fine-tuning
if (IS_FINE_TUNING_ON == True):
    from src.utils.get_best_model import find_best_model

    model_path = find_best_model()
    model_path = os.path.abspath(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)  # Loads best model from the saved checkpoints
    print(f"\n\nModel with the best weights has been loaded from the '{model_path}' archive.")

# Otherwise, it fetches the model weights and the tokenizer configurations from Google Drive
else:
    print("### Downloading model from Google Drive... ###")
    
    downloader.start_downloading_model()
    model_path = "../model/model_version_#5_loss_0.3641332676261663"
    model_path = os.path.abspath(model_path)
    model = AutoModelForSequenceClassification.from_pretrained(model_path, local_files_only=True)  # Loads downloaded model
    print(f"\n\nModel with current best weights has been loaded, located at folder 'model/ ")

### Downloading model from Google Drive... ###


Downloading...
From (original): https://drive.google.com/uc?id=1dxPTErcLS3_fYafjjkzSz_wpE844IjYv
From (redirected): https://drive.google.com/uc?id=1dxPTErcLS3_fYafjjkzSz_wpE844IjYv&confirm=t&uuid=77001a25-939e-4b93-8495-08f12d402c7a
To: /home/titus/projects/roberta-ai-text-detector/model/model_version_#4_loss_0.3737811133265495.zip
100%|██████████| 464M/464M [01:38<00:00, 4.71MB/s] 




Model with current best weights has been loaded, located at folder 'model/ 


# Section 5 - Model Evaluation

In [13]:
# Initialises the test_loader argument for the Eval step
test_loader = DataLoader(X_test, batch_size=16, shuffle=False)                     # Initialises the DataLoader with test set


from src.test_evaluator import TestEvaluator                                       # Imports the TestEvaluator class
evaluator = TestEvaluator(test_loader, model, device)                              # Initialises the evaluator with arguments from above

evaluator.calculate_metrics()                                                      # Starts calculating metrics including: Accuracy, Precision, Recall, F1 score

Evaluating the X_test dataset..
 EVALUATION COMPLETE | OVERALL ACCURACY: 81.88%
                              precision    recall  f1-score   support

Pure Human Written (Class 0)     0.8559    0.4750    0.6109       200
   Pure AI Written (Class 1)     0.9684    0.9200    0.9436       200
       AI Polished (Class 2)     0.6319    0.9100    0.7459       200
         Humanised (Class 3)     0.9194    0.9700    0.9440       200

                    accuracy                         0.8187       800
                   macro avg     0.8439    0.8188    0.8111       800
                weighted avg     0.8439    0.8187    0.8111       800



# Section 6 - Single Inference with Confidence Thresholding

In [18]:
# Custom text for inference
raw_text_input = """
Content detection is an important concern in education, workplaces and public communication.
And, binary classification (AI, human) does not capture what actually happens.
A student creates a report with AI and then edits it by hand. That is different from a lawyer
negotiating a contract and asking AI to fix grammar. There is a spectrum. Both exist. To force them
into one of the two categories doesn’t work.
Instead, we propose a three-class system: AI-generated, human-written, and uncertain (Zhang et al.,
2024). The uncertain class matters because the model should flag when it’s not confident, not
pretend it knows. We fine-tune a RoBERTa model from Hugging Face to handle these three
categories. We test it against real scenarios: edited text, spelling errors, different lengths, formally
rewritten passages; to see where it fails. We report confidence scores alongside predictions so a
person can look at edge cases and not just accept what the model says.
This is a design that works with what detection can actually do."""

from src.inference import inference_pipeline

inference_pipeline( raw_text_input,
                    text_preprocessor,
                    model,
                    tokenizer,
                    device )

CUSTOM THRESHOLD SINGLE INFERENCE ANALYSIS REPORT

Evaluated Model Confidence : 80.72%
Final Categorized Prediction: Class 0 : Pure Human Written


# Section 7 - Visualisations

In [15]:

from src.loss_visualiser import plot_loss_curves

# from loss files extract those arguments for the function
print("### Downloading loss logs from Google Drive... ###")
    
downloader.start_downloading_logs()
print(f"\n\nLogs are fetched and located at folder logs/loss/ ")

### Downloading loss logs from Google Drive... ###


AttributeError: 'Downloader' object has no attribute 'logs'

In [ ]:
train_loss_path = "../logs/loss/model_version_#5_loss_0.03069869413933096"
valid_loss_path = os.path.abspath(model_path)
print(f"\n\nModel is fetched and located at the 'model/' folder.")

plot_loss_curves(10,train_loss_path, valid_loss_path )